In [ ]:
pip install pyfinviz

**Scraping from Screener Page**

In [ ]:
from pyfinviz.screener import Screener

# with no params (default screener table)
#screener = Screener()
# with params (The first 3 pages of "STOCKS ONLY" where Analyst recommend a strong buy)
# print(dir(Screener.BLABLA)) to get info on which attributes does a class have
options = [Screener.IndustryOption.STOCKS_ONLY_EX_FUNDS, Screener.MarketCapOption.LARGE_OVER_USD10BLN]

screener = Screener(filter_options=options, view_option=Screener.ViewOption.VALUATION,
                    pages=[x for x in range(1, 44)])
screenerOV = Screener(filter_options=options, view_option=Screener.ViewOption.OVERVIEW,
                      pages=[x for x in range(1,44)])
screenerF = Screener(filter_options=options, view_option=Screener.ViewOption.FINANCIAL,
                      pages=[x for x in range(1,44)])
screenerP = Screener(filter_options=options, view_option=Screener.ViewOption.PERFORMANCE,
                      pages=[x for x in range(1,44)])
screenerT = Screener(filter_options=options, view_option=Screener.ViewOption.TECHNICAL,
                      pages=[x for x in range(1,44)])

# available variables:
#print(screener.main_url)  # scraped URL
#print(screener.soups)  # beautiful soup object per page {1: soup, 2: soup, ...}
print(screener.data_frames)  # table information in a pd.DataFrame object per page {1: table_df, 2, table_df, ...}

**Dataset Creation**

In [81]:
import pandas as pd
combined_dataOV = pd.concat(screenerOV.data_frames.values(), ignore_index=True)
combined_data = pd.concat(screener.data_frames.values(), ignore_index=True)
combined_dataF = pd.concat(screenerF.data_frames.values(), ignore_index=True)
combined_dataP = pd.concat(screenerP.data_frames.values(), ignore_index=True)
combined_dataT = pd.concat(screenerT.data_frames.values(), ignore_index=True)
print(f"The combined dataset has {combined_dataOV.shape[0]} rows and {combined_dataOV.shape[1]} columns.")
print(f"The combined dataset has {combined_data.shape[0]} rows and {combined_data.shape[1]} columns.")
print(f"The combined dataset has {combined_dataF.shape[0]} rows and {combined_dataF.shape[1]} columns.")
print(f"The combined dataset has {combined_dataP.shape[0]} rows and {combined_dataP.shape[1]} columns.")
print(f"The combined dataset has {combined_dataT.shape[0]} rows and {combined_dataT.shape[1]} columns.")

# Perform successive merges with unique suffixes for each dataset
merged_data = pd.merge(combined_dataOV, combined_data, on="Ticker", how="outer", suffixes=('_OV', '_C'))
merged_data = pd.merge(merged_data, combined_dataF, on="Ticker", how="outer", suffixes=('', '_F'))
merged_data = pd.merge(merged_data, combined_dataP, on="Ticker", how="outer", suffixes=('', '_P'))
merged_data = pd.merge(merged_data, combined_dataT, on="Ticker", how="outer", suffixes=('', '_T'))

# Print the shape of the final merged dataset
print(f"The final merged dataset has {merged_data.shape[0]} rows and {merged_data.shape[1]} columns.")

# Clean dataset
print(merged_data.columns)
# List of columns to drop
columns_to_drop = [
    'No_C', 'MarketCap_C', 'PE_C', 'Price_C', 'Change_C', 'Volume_C',
    'No', 'MarketCap', 'Price', 'Change', 'Volume',
    'No_P', 'Price_P', 'Change_P', 'Volume_P',
    'No_T', 'Price_T', 'Change_T', 'Volume_T'
]

# Drop the specified redundant columns
merged_data = merged_data.drop(columns=columns_to_drop)

# Rename columns by dropping the "_OV" suffix
columns_to_rename = {
    'No_OV': 'No',
    'MarketCap_OV': 'MarketCap',
    'PE_OV': 'PE',
    'Price_OV': 'Price',
    'Change_OV': 'Change',
    'Volume_OV': 'Volume'
}

# Apply the renaming
merged_data.rename(columns=columns_to_rename, inplace=True)

# Print the updated column names to confirm the changes
print("Updated column names:")
print(merged_data.columns)

# Print the shape of the cleaned dataset to confirm the changes
print(f"The cleaned dataset has {merged_data.shape[0]} rows and {merged_data.shape[1]} columns.")

# Save the merged dataset to a CSV file
dataset = "merged_data.csv"  # Specify the desired file name
merged_data.to_csv(dataset, index=False)
print(f"Merged data saved as '{dataset}' in the current working directory.")

The combined dataset has 860 rows and 11 columns.
The combined dataset has 860 rows and 18 columns.
The combined dataset has 860 rows and 18 columns.
The combined dataset has 860 rows and 16 columns.
The combined dataset has 860 rows and 15 columns.
The final merged dataset has 860 rows and 74 columns.
Index(['No_OV', 'Ticker', 'Company', 'Sector', 'Industry', 'Country',
       'MarketCap_OV', 'PE_OV', 'Price_OV', 'Change_OV', 'Volume_OV', 'No_C',
       'MarketCap_C', 'PE_C', 'FwdPE', 'PEG', 'PS', 'PB', 'PC', 'PFCF',
       'EPSthisY', 'EPSnextY', 'EPSpast5Y', 'EPSnext5Y', 'Salespast5Y',
       'Price_C', 'Change_C', 'Volume_C', 'No', 'MarketCap', 'Dividend', 'ROA',
       'ROE', 'ROI', 'CurrR', 'QuickR', 'LTDebtEq', 'DebtEq', 'GrossM',
       'OperM', 'ProfitM', 'Earnings', 'Price', 'Change', 'Volume', 'No_P',
       'PerfWeek', 'PerfMonth', 'PerfQuart', 'PerfHalf', 'PerfYear', 'PerfYTD',
       'VolatilityW', 'VolatilityM', 'Recom', 'AvgVolume', 'RelVolume',
       'Price_P', 'Chang